In [2]:
model = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.2:1b",
    base_url="http://localhost:11434",
    temperature=0,
)

In [2]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434",
)

In [3]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_collection(name="ai_model_book")

# Create chromadb
vector_db_from_client = Chroma(
    client=client,
    collection_name="ai_model_book",
    embedding_function=embedding_model,
)

print("Gespeicherte Textabschnitte:", collection.count())

Gespeicherte Textabschnitte: 2680


In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = (
    {"docs": format_docs}
    | prompt
    | model
    | StrOutputParser()
)

In [5]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(
    search_query,
    k=3,
)

In [6]:
print(chain.invoke(docs))

The main themes in the retrieved documents are:

1. **Classification of Machine Learning Systems**: The documents discuss how to classify Machine Learning (ML) systems into broad categories based on their training methods, learning paradigms, and characteristics.
2. **Types of Supervised Learning**: The documents highlight the four major categories of supervised learning: supervised learning, unsupervised learning, semisupervised learning, and Reinforcement Learning.
3. **Characteristics of Supervised Learning**: The documents emphasize the importance of understanding the amount and type of supervision during training, which is crucial for developing effective ML systems.
4. **Comparison of Supervised and Unsupervised Learning**: The documents compare and contrast supervised and unsupervised learning, highlighting the differences in how the algorithm learns from data and how it is used to make predictions.
5. **Online vs. Batch Learning**: The documents discuss the distinction between 

In [7]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)
print()

The main themes in the retrieved documents are:

1. **Classification of Machine Learning Systems**: The documents discuss how to classify Machine Learning (ML) systems into broad categories based on their training methods, learning paradigms, and characteristics.
2. **Types of Supervised Learning**: The documents highlight the four major categories of supervised learning: supervised learning, unsupervised learning, semisupervised learning, and Reinforcement Learning.
3. **Characteristics of Supervised Learning**: The documents emphasize the importance of understanding the amount and type of supervision during training, which is crucial for developing effective ML systems.
4. **Comparison of Supervised and Unsupervised Learning**: The documents compare and contrast supervised and unsupervised learning, highlighting their differences in terms of data, algorithms, and applications.
5. **Online vs. Batch Learning**: The documents discuss the distinction between online and batch learning, w

In [8]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)
print()

The main themes in the retrieved documents are:

1. **Classification of Machine Learning Systems**: The documents discuss how to classify Machine Learning (ML) systems into broad categories based on their training methods, learning paradigms, and characteristics.
2. **Types of Supervised Learning**: The documents highlight the four major categories of supervised learning: supervised learning, unsupervised learning, semisupervised learning, and Reinforcement Learning.
3. **Characteristics of Supervised Learning**: The documents emphasize the importance of understanding the amount and type of supervision during training, which is crucial for developing effective ML systems.
4. **Comparison of Supervised and Unsupervised Learning**: The documents compare and contrast supervised and unsupervised learning, highlighting their differences in terms of data, algorithms, and applications.
5. **Online vs. Batch Learning**: The documents discuss the distinction between online and batch learning, w

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [9]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

rag_prompt = ChatPromptTemplate.from_template(prompt_template)

retriever = vector_db_from_client.as_retriever(
    search_kwargs={"k": 3}
)

qa_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

In [11]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x11650dcd0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6'}}, model='llama3.2:1b', temperature=0.0, base_url='http://localhos

In [12]:
question = "What is supervised learning?"

print(qa_rag_chain.invoke(question))

Supervised learning is a type of machine learning where the training data includes the desired solutions, called labels, which are used to train the algorithm to make predictions or decisions.


In [13]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes the desired solutions or labels, allowing the algorithm to learn from the data and make predictions or decisions based on the information provided.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [ ]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)